In [ ]:
"""
Plots figure 5
This notebook calculates roundness errors for 2 roundness metrics (Wadell, std)
between the 3um scans and 1.5um scans. Three samples have scans at both resolutions:
OFF-211, K270, K229. Uses the label mappings output by map_1.5_to_3_data.ipynb
 """

import sys
sys.path.append("../../src")
import numpy as np
import roundness_3d as r3d
from skimage.measure import regionprops
import matplotlib.pyplot as plt
import matplotlib as mpl
import glob
mpl.rcParams['figure.dpi'] = 300

def calc_grain_roundness(labeled_vol, grain_region):
        buffer = 10

        minz, minx, miny, maxz, maxx, maxy = grain_region.bbox
        bin_vol = (labeled_vol[minz-buffer:maxz+buffer,minx-buffer:maxx+buffer,miny-buffer:maxy+buffer] == grain_region.label).astype(np.uint8)
        return r3d.wadell_curvature_estimate(bin_vol), r3d.std_curvature(bin_vol)

def calc_error(lr, hr):
        # order = wadell, std
        norm_ranges = np.array([[0.40459754477234844,0.7922314926916163],[1.148691371175796,4.79610462587357]]) # calculated from locally binned averages from compare_3d_roundnesses.ipynb
        metric_ranges = norm_ranges[:,1] - norm_ranges[:,0]
        if np.any(metric_ranges <= 0):
                print('error in range calculation')
        return (lr - hr) / metric_ranges

# sample info
sample_ids = ['K229','K270','OFF-211'] 

# path to directory of labeled volumes
vol_path = '../../../../processed_data/labeled/'

for sid in sample_ids:

        # load low res and hi res labeled volumes
        hr_file = glob.glob(vol_path + sid + '_1p5um*')
        lr_file = glob.glob(vol_path + sid + '_3um*')

        if len(hr_file) != 1 or len(lr_file) != 1:
                raise FileNotFoundError("Expected exactly one matching file for each scan.")

        hr_vol = np.load(hr_file[0]) # 1.5 micron data
        lr_vol = np.load(lr_file[0]) # 3 micron data

        # load label mappings
        map_path = '../../../../processed_data/3um_to_1.5um_label_maps/'
        label_map = np.load(map_path + sid + 'label_map.npy')

        # set output data filename
        output_data_fn = f'../../../../processed_data/3um_vs_1.5um_round_errors/{sid}_round_errs.npy'

        print(hr_vol.shape)
        print(lr_vol.shape)

        lr_regions = regionprops(lr_vol)
        hr_regions = regionprops(hr_vol)

        # check that index in regions is always label - 1 (sequential labels in each volume)
        n_errors = 0
        for i in range(len(lr_regions)):
                if i != lr_regions[i].label - 1:
                        n_errors += 1
    
        if n_errors != 0:
                print(n_errors)

        n_errors = 0
        for i in range(len(hr_regions)):
                if i != hr_regions[i].label - 1:
                        n_errors += 1
    
        if n_errors != 0:
                print(n_errors)

        lr_rounds = np.zeros((len(lr_regions), 2))
        hr_rounds = np.zeros((len(lr_regions), 2))

        # loop through low-res grains
        for i, lr_grain in enumerate(lr_regions):
                lr_label = lr_grain.label
                ind = np.nonzero(lr_label == label_map[:,0])[0]
                if len(ind) == 0:
                        continue
                
                lr_rounds[i] = calc_grain_roundness(lr_vol, lr_grain)

                hr_label = label_map[ind[0],1]
                hr_grain = hr_regions[hr_label - 1]

                hr_rounds[i] = calc_grain_roundness(hr_vol, hr_grain)

        # remove all values where = 0 (corresponding to grains not in hr scan
        # also remove all values where either lr_rounds or hr_rounds was -100,
        # which is the value set for grains that are too small
        is_valid = np.logical_not(np.logical_or(np.any(lr_rounds <= 0, axis=1), np.any(hr_rounds <= 0, axis=1)))

        lr_rounds = lr_rounds[is_valid]
        hr_rounds = hr_rounds[is_valid]
        round_errors = calc_error(lr_rounds, hr_rounds)

        # save error data
        np.save(output_data_fn, round_errors)

In [ ]:
# imports
import numpy as np
import sys
import matplotlib.pyplot as plt
sys.path.append('../../../pys')
from fplot import format_plots

CR, CS, CE, CL, CRD, CED = format_plots() # format plots and collect colors
colors = ['#f99d1b', "#205073"] # wadell, std_curv

# merge all data together
sample_ids = ['K229','K270','OFF-211'] 
error_data_dir = f'../../../../processed_data/3um_vs_1.5um_round_errors/'

round_errs_1 = np.load(error_data_dir+f'{sample_ids[0]}_round_errs.npy')
round_errs_2 = np.load(error_data_dir+f'{sample_ids[1]}_round_errs.npy')
round_errs_3 = np.load(error_data_dir+f'{sample_ids[2]}_round_errs.npy')

all_errors = np.concatenate((round_errs_1, round_errs_2, round_errs_3), axis=0)

print(all_errors.shape)

fig, axes = plt.subplots(2, 1, figsize=(3.46, 4),sharex=True,sharey=True)
axes = axes.flatten()
metric_names = ['Wadell', 'Std']
round_metrics = [r'$R_{W,\kappa}$',r'$R_{\sigma,\kappa}$']

bin_size = 0.1
bins = np.arange(-0.55, 0.56, bin_size) # just from looking at data max and min; if new data, let it set default for new data and then reset bins

axes[1].set_xlabel('High vs low resolution metric disparity\n(normalized by metric range)')
for i in range(2):
    print(all_errors[:, i].max())
    print(all_errors[:, i].min())
    lbl = f'{round_metrics[i]}\n$\mu=${np.mean(all_errors[:, i]):.2f}\n$\sigma=${np.std(all_errors[:, i]):.2f}'
    # axes[i].axvline(x=0, color="#565656", linestyle='--', linewidth=1)
    axes[i].hist(all_errors[:, i], bins, color=colors[i],edgecolor='black', label=lbl)
    axes[i].axvline(x=np.mean(all_errors[:, i]), color='tab:red', linestyle='--', linewidth=1)
    axes[i].set_ylabel('Frequency')
    axes[i].set_xlim(-0.6,0.6)
    axes[i].legend()

plt.tight_layout()

plt.savefig('../plots/case_study_hi_vs_lo_res.pdf')
plt.show()